In [1]:
from utils import dataloader

import yaml

import numpy as np
import pandas as pd

from tqdm import tqdm

import nibabel as nib

/home/petron/AIMS/Thesis/TRACE_Algonauts/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


[fetch_atlas_schaefer_2018] Dataset found in /home/petron/nilearn_data/schaefer_2018


In [2]:
with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

#### Get File Details

In [ ]:
subject = 1

fmri_stimuli_info = dataloader.list_fmri_sessions(
    dir=FMRI_DIR,
    subject=subject,
    split='train'
)

In [ ]:
fmri_stimuli_info

#### Load Transcript

In [ ]:
fmri_stimuli_info

In [34]:
fmri_stimuli_info[0][13:-2]

's01e0'

In [44]:
transcript_df = dataloader.load_transcript(
    dir = TRANSCRIPT_DIR, 
    stimuli_name=fmri_stimuli_info[0],
    split='train',
    ignore_nans=True
    )

transcript_df.shape

(366, 4)

In [45]:
transcript_df

,text_per_tr,words_per_tr,onsets_per_tr,durations_per_tr
0,You.,['You.'],[0.33],[0.55]
1,What you guys don't,"['What', 'you', 'guys', ""don't""]","[1.89, 2.228, 2.388, 2.612]","[0.316, 0.138, 0.186, 0.214]"
2,"understand is, for us,","['understand', 'is,', 'for', 'us,']","[2.858, 3.22, 3.556, 3.812]","[0.292, 0.282, 0.218, 0.586]"
3,kissing is as important,"['kissing', 'is', 'as', 'important']","[4.564, 5.098, 5.268, 5.428]","[0.502, 0.148, 0.138, 0.33]"
4,as any part of it.,"['as', 'any', 'part', 'of', 'it.']","[5.844, 6.068, 6.292, 6.468, 6.628]","[0.202, 0.186, 0.154, 0.138, 0.522]"
...,...,...,...,...
473,She's,"[""She's""]",[705.37],[0.68]
474,pregnant with my,"['pregnant', 'with', 'my']","[706.13, 706.706, 707.036]","[0.512, 0.276, 0.218]"
475,child.,['child.'],[707.292],[0.588]
476,And she and Susan are going,"['And', 'she', 'and', 'Susan', 'are', 'going']","[708.97, 709.436, 709.628, 709.788, 710.178, 7...","[0.412, 0.17, 0.138, 0.358, 0.196, 0.154]"


#### Load h5

In [37]:
subject = 5

recording_session = '001'
season = '01'
episode = '02'
episode_split = 'b'

fmri_key = f'ses-{recording_session}_task-s{season}e{episode}{episode_split}'
fmri_key

'ses-001_task-s01e02b'

In [ ]:
file_name = "sub-01_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5"
fmri_data = dataloader.load_fmri_responses(
    dir=FMRI_DIR,
    subject=subject,
    stimuli_name=fmri_key,
    split='train'
)
fmri_data.shape

(482, 1000)

# Load Atlas Info

In [3]:
import yaml

with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

with open('configs/configs.yaml', 'r') as f:
    configs = yaml.safe_load(f)

hrf_delay = configs['hrf_delay']
tr = configs['tr']
context_trs = configs['context_trs']

subjects = configs['subjects']
subjects

[1, 2, 3, 5]

In [ ]:
subject1_atlas_coord, parcel_ids = dataloader.load_atlas_for_subject(subject=1, dir=FMRI_DIR)
subject2_atlas_coord, parcel_ids = dataloader.load_atlas_for_subject(subject=2, dir=FMRI_DIR)
subject3_atlas_coord, parcel_ids = dataloader.load_atlas_for_subject(subject=3, dir=FMRI_DIR)
subject5_atlas_coord, parcel_ids = dataloader.load_atlas_for_subject(subject=5, dir=FMRI_DIR)

In [24]:
(subject1_atlas_coord == subject2_atlas_coord).sum()

np.int64(3000)

# Movie Scenes Response Matrix

In [3]:
import yaml

with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

with open('configs/configs.yaml', 'r') as f:
    configs = yaml.safe_load(f)

hrf_delay = configs['hrf_delay']
tr = configs['tr']
context_trs = configs['context_trs']

subjects = configs['subjects']
subjects

[1, 2, 3, 5]

#### season-episode-split as trials

In [ ]:
subject_dataset = {}

for subject in subjects:
    print(f"Loading for s-0{subject}")
    fmri_stimuli_info = dataloader.list_fmri_sessions(
        dir = FMRI_DIR,
        subject = subject,
        split = 'train'
    )

    for stimuli in tqdm(fmri_stimuli_info, total=len(fmri_stimuli_info)):
        stimuli_fmri_response = dataloader.load_fmri_responses(
            dir = FMRI_DIR,
            subject = subject, 
            stimuli_name = stimuli,
            split='train'
        )
        if stimuli not in subject_dataset:
            subject_dataset[stimuli] = [stimuli_fmri_response]
        else:
            subject_dataset[stimuli].append(stimuli_fmri_response)

for stimuli in subject_dataset:
    subject_dataset[stimuli] = np.stack(subject_dataset[stimuli])

Loading for s-01


100%|██████████| 292/292 [00:02<00:00, 113.39it/s]


Loading for s-02


100%|██████████| 292/292 [00:02<00:00, 102.45it/s]


Loading for s-03


100%|██████████| 290/290 [00:02<00:00, 101.12it/s]


Loading for s-05


100%|██████████| 289/289 [00:02<00:00, 100.44it/s]


In [ ]:
subject_dataset['ses-001_task-s01e02a'].shape

(4, 482, 1000)

#### Transcripts based trials

In [ ]:
subject = subjects[0]
SPLIT = 'train'

print(f"Loading for s-0{subject}")
fmri_stimuli_info = dataloader.list_fmri_sessions(
    dir = FMRI_DIR,
    subject = subject,
    split = SPLIT
)

scenes_response_matrix = {}

# print(fmri_stimuli_info)

for stimuli in tqdm(fmri_stimuli_info, total=len(fmri_stimuli_info)):
    stimuli_fmri_response = dataloader.load_fmri_responses(
        dir = FMRI_DIR,
        subject = subject, 
        stimuli_name = stimuli,
        split=SPLIT
    )

    trials, start, end = dataloader.epoch_fmri_by_words(
        stimuli, stimuli_fmri_response, 
        TRANSCRIPT_DIR, FMRI_DIR,
        hrf_delay, tr, context_trs,
        split=SPLIT
    )
    if trials is None:
        continue

    # print(trials.shape, start.shape, end.shape)
    stimuli_name = stimuli[13:]
    if stimuli not in scenes_response_matrix:
        scenes_response_matrix[stimuli_name] = {
            'trials': [trials],
            'start': [start],
            'end': [end]
        }
    else:
        scenes_response_matrix[stimuli_name]['trials'].append(trials)
        scenes_response_matrix[stimuli_name]['start'].append(start)
        scenes_response_matrix[stimuli_name]['end'].append(end)

for stimuli_name in scenes_response_matrix:
    scenes_response_matrix[stimuli_name]['trials'] = np.stack(scenes_response_matrix[stimuli_name]['trials'], axis=0).squeeze(0)
    scenes_response_matrix[stimuli_name]['start'] = np.stack(scenes_response_matrix[stimuli_name]['start'], axis=0).squeeze(0)
    scenes_response_matrix[stimuli_name]['end'] = np.stack(scenes_response_matrix[stimuli_name]['end'], axis=0).squeeze(0)


## Load Subject Parcel Coordinates
print(f"Loading for s-0{subject} parcel coordinates")
parcel_coords, parcel_ids, parcel_desc = dataloader.load_atlas_for_subject(subject=subject, dir=FMRI_DIR)

subject_dataset = {
    "scenes_response": scenes_response_matrix,
    "parcel_coords": parcel_coords,
    "parcel_ids": parcel_ids,
    "parcel_desc": parcel_desc
}

Loading for s-01


100%|██████████| 292/292 [00:15<00:00, 19.36it/s]


Loading for s-01 parcel coordinates


In [5]:
subject_dataset.keys()

dict_keys(['scenes_response', 'parcel_coords', 'parcel_ids', 'parcel_names'])

In [ ]:
subject_dataset['parcel_desc']

[{'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},
 {'hemisphere': 'LH', 'region': 'Vis'},


In [ ]:
for stimuli_name in subject_dataset.keys():
    print(
        subject_dataset[stimuli_name]['trials'].shape, 
        (subject_dataset[stimuli_name]['start'].shape, subject_dataset[stimuli_name]['end'].shape), 
        (subject_dataset['st']))

(360, 10, 1000) (360,) (360,)
(328, 10, 1000) (328,) (328,)
(347, 10, 1000) (347,) (347,)
(302, 10, 1000) (302,) (302,)
(389, 10, 1000) (389,) (389,)
(362, 10, 1000) (362,) (362,)
(364, 10, 1000) (364,) (364,)
(323, 10, 1000) (323,) (323,)
(454, 10, 1000) (454,) (454,)
(433, 10, 1000) (433,) (433,)
(384, 10, 1000) (384,) (384,)
(363, 10, 1000) (363,) (363,)
(356, 10, 1000) (356,) (356,)
(352, 10, 1000) (352,) (352,)
(314, 10, 1000) (314,) (314,)
(334, 10, 1000) (334,) (334,)
(325, 10, 1000) (325,) (325,)
(355, 10, 1000) (355,) (355,)
(375, 10, 1000) (375,) (375,)
(363, 10, 1000) (363,) (363,)
(345, 10, 1000) (345,) (345,)
(363, 10, 1000) (363,) (363,)
(301, 10, 1000) (301,) (301,)
(318, 10, 1000) (318,) (318,)
(338, 10, 1000) (338,) (338,)
(338, 10, 1000) (338,) (338,)
(333, 10, 1000) (333,) (333,)
(340, 10, 1000) (340,) (340,)
(355, 10, 1000) (355,) (355,)
(328, 10, 1000) (328,) (328,)
(365, 10, 1000) (365,) (365,)
(363, 10, 1000) (363,) (363,)
(371, 10, 1000) (371,) (371,)
(316, 10, 